# report07 — SBR — 표적을 조준해 밝기를 계산하다

**핵심.** report06 이 '경로 솔버로는 못 만든다' 고 못박은 표적 밝기(RCS)를, **Sionna 가 쓰는 Mitsuba 광선 위에 표준 PO 표면적분을 얹은 SBR+PO 로 계산한다** — 답을 아는 평판·금속구 해석해로 교정하면서.

| 이 리포트의 척추 |  |
|---|---|
| **① Sionna 의 공백** | Sionna RT `PathSolver` 에는 표면 산란적분 단계가 없어 표적 밝기 σ 를 주지 못한다(report06 이 다섯 측정으로 확인). 밝기는 표면 조각들의 되쏨을 위상까지 맞춰 다 더한 값인데, 표적을 거울로만 튕기는(GO) 경로 솔버에는 그 ∫ 단계가 없다. |
| **② 선행 연구의 방식** | 소형 표적의 코히어런트 RCS 를 스톡 Sionna 로 낸 선행은 **없고**, ISAC 문헌은 표적 밝기를 외부에서 구해 채널에 주입한다 — **(b)** 재질 확산계수 S 가정[Great-X, arXiv:2507.08716], **(c)** 상용 full-wave 로 계산·주입 $h=h_{bg}+h_{target}$[LAMBDA=Sionna+CADFEKO, arXiv:2607.03826; Temporal-GNN=점산란체, arXiv:2604.08306], **(d)** 자작 산란 add-on[Ziganshin=Sionna-RT+UTD, arXiv:2604.05991] (§2). |
| **③ 쓴 라이브러리·결합** | **(d)** 를 택했다 — 새 광선엔진을 만들지 않는다. Sionna 가 쓰는 **Mitsuba 3 / OptiX 광선을 그대로 재사용**(중복계산 회피)하고 그 위에 **자작 PO 표면적분**(`src/rcs_sbr.py`, 가림 포함)만 얹는다. GPU BVH SBR+PO(arXiv:2604.09243)와 **같은 계열**이다(적용범위는 다르다 — 그들은 PEC·모노스태틱 후방산란 전용, 우리는 다중재질·바이스태틱; report06 §4)(§3). |
| **④ 검증** | 답을 아는 정준 표적에 **단일 입사방향 후방산란**으로 대고 재본다 — 금속 평판 σ=4πA²/λ² 오차 **-0.01 dB**(정면입사라 위상항이 상수 → 이건 *면적 구적* 확인이지 위상·파장 검증이 아니다), 곡면인 금속구 σ=πr² 오차 **+0.39 dB** @λ/10 이지만 격자밀도에 **단조롭지 않아** λ/12 에서 +1.45 dB, 우리 설정 λ/16 에서 -0.58 dB (@3.5 GHz). report6 원장의 -0.01/+0.39 dB 는 **같은 커널을 같은 인자로 다시 부른 회귀 확인**이지 독립 재현이 아니다(§4). 절대값 앵커는 report08. |

---


## 📋 이 결과가 어디서 어떻게 나왔나

> 이 절은 **직접 참여하지 않은 사람도 출처를 따라가고 재현할 수 있도록** 넣었습니다. 버전·GPU 는 노트북 생성 시점에 **실제로 읽어온 값**입니다.

### 1️⃣ 무엇을 참고했나

| 항목 | 출처 | 성격 |
|---|---|---|
| 검증 기준 (정답) | 교과서 폐형식 — 금속구 σ = πr² (광학영역) · 금속평판 σ = 4πA²/λ² (정면 조사) | 📐 해석해 |
| SBR 검증·격자수렴·가림 측정값 | **`outputs/report2_waveform_rcs.json`** 의 `sbr_validation` / `occlusion` (그림 `report2_sbr_validate.png` · `report2_occlusion.png` 과 같은 소스) | 🟡 측정 (SBR = 우리 구현, Mitsuba 광선) |
| 같은 커널의 회귀(재현) 확인 | **`outputs/report6_sbr.json`** 의 `kernel` — 다른 스크립트에서 **같은 함수를 같은 인자로** 다시 부른 값(`viz_report2.py:588-589` ↔ `viz_verify_sbr.py:107-108`). 결정론적 동일 코드경로라 값이 같은 것이 당연하다 — **독립 구현 대조가 아니다** | 🟡 회귀 확인 (독립 검증 아님) |
| 광선격자·삼각형 세분 민감도 | **`report_mesh/outputs/mesh_verify.json`** 의 `I_sbr_subdiv` (λ/12→λ/24 재계산 · 삼각형 4배 세분) | 🟡 측정 (report_mesh 원장) |
| SBR 구현 | **`src/rcs_sbr.py`** — Mitsuba 3 / OptiX 광선 + PO 표면적분. **Sionna 가 쓰는 그 광선엔진 그대로** | 🟡 우리 구현 (Sionna 에 RCS 솔버가 없음) |

### 2️⃣ 어떤 도구가 무엇을 했나 — **Sionna 내부인가, 우리가 짠 건가**

| 도구 | 하는 일 | 어디서 도는가 |
|---|---|---|
| `sbr` | SBR (`src/rcs_sbr.py`) — **Mitsuba 광선 + PO 표면적분**으로 RCS. 가림(occlusion) 포함 | 🟡 **우리가 짰다** — 다만 광선추적은 Sionna 가 쓰는 **Mitsuba 3 엔진 그대로** (GPU). Sionna 에 RCS 솔버가 없기 때문 |
| `po` | 순수 물리광학 (`src/rcs_po.py`) — 점구름 PO. **가림 없음** | 🔴 **별도** (numpy, CPU). **비교·검증용으로만** 남겨둠 — 기본 엔진은 SBR |
| `sionna-render` | Sionna RT `Scene.render_to_file()` — 씬·**추적된 광선**·라디오맵을 사진처럼 렌더 | 🟢 **Sionna 내부** (Mitsuba 3 경로추적 렌더러, GPU) |
| `matplotlib` | matplotlib — 도표·그래프 | 🔴 **별도** (CPU). 계산 결과를 *그리기만* 한다 |

> 🔑 **이 구분이 이 프로젝트에서 가장 자주 오해받는 지점입니다.**
> - **전파**(경로·지연·도플러·렌더·라디오맵)는 🟢 **Sionna 가** 합니다.
> - **표적 RCS** 는 🟡 우리가 얹은 **PO(물리광학 표면적분)** 가 냅니다 — Sionna 기본 solver 엔 이 산란적분이 없어 경로 이득만 줄 뿐 RCS 를 못 내기 때문입니다. 광선을 쏴 조명면·가림을 찾는 **SBR** 은 Sionna 의 **Mitsuba 3 엔진을 그대로** 쓰고, 그 위에 **PO 적분만 우리가** 얹습니다(SBR+PO).
> - **레이더 신호처리**(ECA/CFAR)는 🔴 우리가 짰습니다 — Sionna 에 레이더 DSP 가 없습니다.

### 3️⃣ 라이브러리 (실행 시점 **실측** 버전)

| 라이브러리 | 버전 | 무엇에 쓰나 |
|---|---|---|
| `sionna` | 2.0.1 | 광선추적(RT) + PHY(OFDM/NR/채널) — **이 프로젝트의 중심** |
| `mitsuba` | 3.8.0 | Sionna RT 의 렌더러·광선추적 백엔드 (OptiX, GPU). SBR 도 이걸 쓴다 |
| `drjit` | 1.3.1 | Mitsuba 의 JIT 컴파일러 — GPU 커널 생성 |
| `trimesh` | 4.12.2 | 메쉬 CAD·**검증** — 로프트/스윕/불리언 + watertight·법선·퇴화면 검사 |
| `numpy` | 2.5.0 | 수치 계산 전반 |
| `matplotlib` | 3.11.0 | 도표 |

### 4️⃣ 어디서 돌렸나

- **Python** 3.12.13 · Linux 5.15.0-136-generic
- **GPU** — `src/gpu.py` 가 **여유 메모리를 보고 자동 선택**합니다 (하드코딩 없음):
  - 0, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
  - 1, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
  - 2, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
  - 3, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
- `CUDA_VISIBLE_DEVICES` = (고정 안 함 — src/gpu.py 가 여유 메모리 보고 자동 선택)

- **계산 비용**: SBR 커널 검증은 GPU 한 장에서 수십 초. 광선격자 λ/16, Mitsuba 3 / OptiX.

### 5️⃣ 어떻게 다시 돌리나 (재현)

```bash
cd /home/yunjung/workspace/sionna2

# 커널 해석해 검증(구 πr² / 평판 4πA²/λ²) + 격자 수렴
~/.venvs/py312/bin/python src/rcs_sbr.py

# 측정 + 그림 + JSON (sbr_validation / occlusion 을 남긴다)
~/.venvs/py312/bin/python src/viz_report2.py

# 같은 커널을 같은 인자로 다시 부르는 회귀 확인 (report6_sbr.json kernel)
~/.venvs/py312/bin/python src/viz_verify_sbr.py

# JSON -> report07.ipynb (이 파일)
~/.venvs/py312/bin/python src/make_notebook07.py
```

### 6️⃣ 본문 숫자는 어디서 오나

이 노트북의 **숫자는 손으로 적지 않았습니다.** 측정 스크립트가 JSON 을 남기고, 노트북 생성기(`src/make_notebook*.py`)가 그 JSON 을 읽어 본문에 주입합니다. → **그림과 글이 어긋날 수 없습니다.** 숫자가 이상하면 JSON 을 보세요.

### 7️⃣ 무엇이 산출되나

| 산출물 | 무엇 |
|---|---|
| `outputs/report2_waveform_rcs.json` | **이 노트북의 SBR 숫자.** sbr_validation / occlusion 블록 |
| `outputs/report6_sbr.json` | 같은 커널을 같은 인자로 다시 부른 회귀 확인 (kernel 블록) |
| `report_mesh/outputs/mesh_verify.json` | §4 광선격자 민감도 · 삼각형 세분 불변성 (I_sbr_subdiv 블록) |
| `outputs/figures/report2_sbr_validate.png` | §4 SBR 검증 · 격자 수렴 |
| `outputs/figures/report2_occlusion.png` | §5 가림 — 순수 PO vs SBR |
| `outputs/figures/report2_po_vs_sbr.png` | §5 방위 패턴 비교 |

### 8️⃣ ⚠️ 믿으면 안 되는 것 (신뢰 경계)

> 정직함이 이 프로젝트의 규칙입니다. **아래는 이 리포트가 보장하지 않는 것들입니다.**

- **절대 RCS 는 문헌 실측보다 밝은 쪽으로 치우친다 — 신뢰의 중심에 두지 않는다.** SBR 은 해석해(구·평판)로 교정되지만, 절대 dBsm 을 문헌과 견주면 **크기가 맞는 짝끼리 aspect-peak ↔ aspect-peak** 으로 보아 우리 쪽이 위로 치우친다(정량치와 짝짓기 규약은 report08 §6). few-λ(공진영역)에서 PO 가 절대레벨을 밝은 쪽으로 잡는 경향이 유력한 설명이다. 이 리포트가 지지하는 것은 방법의 정합성·상대 패턴이지 절대 dBsm 의 정밀값이 아니며, 검출 결과는 σ 밴드로 제시해 상대 결론의 robust 함을 보인다. ⚠ '소형드론 방위평균 포락선 −28~−16 dBsm' 은 문헌이 직접 보고한 값이 아니라 aspect-peak 과 자세 스프레드에서 우리가 유도한 2차 산물이므로 판정 근거로 쓰지 않는다(report08 §6).
- **격자 위상 진동은 지터평균으로 잡되, 완전히 없어지지는 않는다.** 곡면은 광선격자를 어디에 맞추느냐에 절대레벨이 흔들린다 — 정렬만 흔든 산포가 λ/8 5.28 dB, 우리 설정 λ/16 1.78 dB, λ/24 0.34 dB. 서브셀 오프셋 격자를 평균하면 잔차가 λ/8 -1.19 → λ/16 +0.30 → λ/24 +0.02 dB 로 줄지만, **우리 격자에서는 아직 0.30 dB 남아 있다.** 절대 dBsm 은 측정 앵커에 맡긴다 — §4.
- **오목한 곳의 다중반사(2·3차) 값은 규모(작다)만 신뢰한다.** SBR 이 실제로 고치는 것은 가림이고, 다중반사는 부차적이고 작은 항이다. 그 정확한 dB 값은 이 리포트의 주장이 아니다.
- **유전체 셸 투과는 1차 근사로 반영한다** — 준투명 플라스틱 셸(body·canopy)을 통과시켜 내부 금속(배터리·PCB)을 왕복 투과계수 τ=1−|Γ|² 로 코히런트 합산한다(안 하면 지배 산란체가 삭제되어 σ 가 ~1.3 dB 낮아진다). 단 셸의 굴절 굴곡·유전체 내부 위상지연은 무시하는 1차 투과다.
- **모서리·정점 회절은 비운다.** SBR+PO 는 정반사(specular) 항만 채우고 에지·정점 회절(E_edge·E_vertex)을 계산하지 않아 그림자 영역·깊은 널이 부정확하다. 우리와 가장 가까운 선행(Ziganshin, arXiv:2604.05991)은 SBR+PO 의 이 한계를 명시적으로 비판하며 UTD 로 그 항을 채우지만, 그 유효조건 facet E>1.5λ 는 few-λ 유전체 드론에서 성립하지 않는다 — 자세한 원장은 §6.
- **표적이 밝아야 잡힌다 — RCS 는 디텍션에 필수다.** 다만 그 밝기를 Sionna 기본 광선으론 낼 수 없어(산란적분 부재) SBR 로 따로 계산할 뿐이다.

### 9️⃣ 앞뒤 리포트

| 리포트 | 관계 |
|---|---|
| **앞** — [report06](report06.ipynb) | Sionna 의 한계 — 왜 기본 광선엔진이 표적 밝기(σ)를 못 내나 |
| **다음** — [report08](report08.ipynb) | 이 SBR 로 잰 **실제 드론들의 밝기(RCS) 결과** |

<details><summary><b>🔤 용어집 — 모르는 말이 나오면 여기</b> (클릭)</summary>

| 용어 | 뜻 |
|---|---|
| **RCS (σ)** | 레이더 되비침 밝기 [m²]. 표적이 레이더 쪽으로 얼마나 세게 되쏘는가. 밝을수록 잡기 쉽다. dBsm = 10·log₁₀(σ / 1 m²) |
| **σ 를 넓이로** | 밝기는 결국 **되쏘는 표면의 넓이**에서 나온다 — 넓은 판이 작은 못보다 밝게 반짝인다 |
| **dBsm** | 1 m² 를 기준으로 한 데시벨 단위. −20 dBsm ≈ 0.01 m² |
| **PO (물리광학)** | 표적 표면의 작은 조각들이 되쏘는 양을 **위상까지 맞춰 다 더하는 계산**. 밝기(σ)는 이 덧셈에서 나온다 |
| **SBR** | Shooting-and-Bouncing Rays. 표적을 **조준해 광선을 쏘고**, 맞은 면이 레이더로 **되쏘는 양을 PO 로 계산·합산**한다. 상용 전자기 솔버의 표준 |
| **GO (기하광학)** | 표면을 '점 거울' 로만 본다. 벽·바닥에서 어디로 튕기는지는 정확하지만 **넓이 항이 없어 밝기를 못 준다** — 전파 광선추적이 이것 |
| **가림(occlusion)** | 앞의 면에 막혀 실제로는 레이더에 안 보이는 면. 이걸 안 빼면 밝기를 부풀린다 |
| **정반사점** | 광선이 거울처럼 **똑바로 되돌아오는 딱 한 지점**. 매끈한 곡면은 이게 하나뿐 |
| **격자밀도 λ/d** | 쏘는 평행 광선을 파장(λ)당 몇 발로 촘촘히 하느냐. λ/16 = 파장 한 칸에 16 발 |
| **위상** | 파동이 되쏠 때의 타이밍. 같은 타이밍으로 겹치면 밝아지고 엇갈리면 상쇄된다 |

</details>

---


## §1. Sionna 의 공백 — 기본 광선엔진은 표적 밝기를 못 낸다

레이더는 결국 **되돌아온 메아리의 세기**로 표적을 알아챈다. 표적이 어두우면(RCS 가 작으면) 메아리가 주변 잡음에 묻혀 검출이 실패한다. 그러니 **표적이 얼마나 밝게 되비추는가**(RCS, σ; dBsm = 10·log₁₀(σ/1 m²))를 아는 것이 디텍션의 출발점이다.

밝기는 표면 조각들이 되쏘는 파동을 **위상까지 맞춰 다 더한(산란적분)** 값이다 — 넓은 판이 작은 못보다 밝게 반짝이듯, 밝기는 되쏘는 표면 넓이에서 나온다. 그런데 전파 시뮬레이터의 기본 광선엔진(Sionna RT `PathSolver`)은 광선이 벽·바닥에서 **어디로 튕기는지**는 정확히 알지만, 표적을 국소 거울(GO)로만 튕겨 그 **넓이 항(∫ 산란적분)이 식에 없다.** 그래서 σ 를 주지 못한다.

이것은 우리만의 진단이 아니라 **NVIDIA 가 직접 확인**한 사실이다 — Sionna RT 창설논문(arXiv:2303.11103)은 이 솔버가 경로별 복소이득만 반환한다고 명시하고, Sionna 개발 메인테이너(J. Hoydis)는 커뮤니티 질문('광선을 대량 쏴 금속체 RCS 를 낼 수 있나')에 **"This is currently not supported"**(GitHub Discussion #844)라고 답했다. 즉 상세 mesh 표적 RCS 는 Sionna 기본 기능이 아니라 **사용자가 직접 얹어야 하는 부분**이다. **앞 리포트 [report06](report06.ipynb) 이 이 공백을 광선 4억 발·금속구·평판·재질 스윕 다섯 측정으로 확인**했다. 이 리포트는 그 빈자리를 **어떤 표준 방법으로 메웠나**로 바로 간다.

---
## §2. 선행 연구는 이 공백을 어떻게 채웠나 — 세 갈래

소형 표적의 코히어런트 RCS 를 스톡 Sionna 로 메쉬에서 직접 낸 선행은 **없다.** Sionna 를 센싱(ISAC)에 쓰는 연구들은 표적 밝기를 **외부에서 구해 채널에 주입**하되, 그 '외부' 를 세 갈래로 채운다:

| 갈래 | 방식 | 대표 선행 |
|---|---|---|
| **(b)** | 재질 **확산계수 S** 가정 | Great-X (arXiv:2507.08716) |
| **(c)** | 상용 full-wave 로 계산·주입 $h=h_{bg}+h_{target}$ | LAMBDA=Sionna+CADFEKO(arXiv:2607.03826) · Temporal-GNN=점산란체(arXiv:2604.08306) |
| **(d)** | 자작 **산란 add-on**(SBR+PO·UTD) | Ziganshin=**Sionna-RT+UTD**(arXiv:2604.05991) · GPU BVH SBR+PO(arXiv:2604.09243·독립엔진) |

<sub>같은 (d) 안에서도 **붙이는 층이 다르다** — Ziganshin 은 Sionna-RT 솔버 자체를 확장해 UTD 회절을 그 안에 넣고, 우리는 Sionna 가 쓰는 Mitsuba 광선엔진 위에 PO 표면적분을 따로 얹는다(`PathSolver` 는 상속 확장점이 아니다 — report06 §3).</sub>

공통 아키텍처는 $h_{surv}=h_{direct}+h_{background}+h_{target}$ — 환경 전파는 Sionna 가 주고, 표적 산란만 외부 물리로 계산해 두 전파 구간 사이에 끼운다. 이 중 **(d) SBR+PO** 는 FEKO·CST 같은 상용 솔버가 큰 표적에 쓰는 표준이자, 최근 ISAC 연구(GPU BVH SBR+PO, arXiv:2604.09243)가 메쉬에서 직접 RCS 를 내는 바로 그 방법이다. 우리는 **(d) 로 값을 계산해 (c) 로 주입**하는 길을 택한다 — 상용툴(CADFEKO)은 유료라 재현이 막히고, 공개 SBR 도구(RaytrAMP, GPL-3.0)는 바이스태틱+다중재질 요구를 못 채우기 때문이다. 다음 §3 이 그 (d) 가 실제로 무엇을 하는지 보인다.

---
## §3. 우리가 쓴 방식 — Mitsuba 광선 위의 SBR+PO

SBR(Shooting-and-Bouncing Rays)은 밝기를 내기 위해 **두 가지**를 한다.

**① 사방에 뿌리지 않고 표적을 정면 조준한다.** 광선을 아무 방향으로나 흩뿌리면 작은 표적은 대부분 빗나간다. 대신 시선 방향 $\hat u$ 에서 **간격이 촘촘한 평행 광선 격자**를 표적에 곧장 쏜다. 그러면 거의 모든 광선이 표적의 어딘가에 맞는다.

**② 물리적으로 되튕겨 오길 기다리지 않고, 되쏘는 양을 그 자리에서 계산한다.** 광선이 맞은 점 $\vec p_i$ 마다, 그 면이 레이더 쪽으로 되쏘는 양을 PO(물리광학)로 — **위상까지 맞춰** — 셈해 전부 더한다:

$$E(\hat u)=\sum_{\text{맞은 점}} |\Gamma_i|\; e^{\,j\,2k\,\vec p_i\cdot\hat u}\; d^2,\qquad \sigma=\frac{4\pi}{\lambda^2}\,|E|^2$$

- $|\Gamma_i|$ = 그 점 재질이 되쏘는 세기(반사계수), $d$ = 광선 격자 간격($d^2$ = 광선 한 발이 대표하는 넓이), $k=2\pi/\lambda$ = 파수.
- $e^{\,j2k\,\vec p_i\cdot\hat u}$ 는 **위상**이다 — 각 점의 되쏨이 같은 타이밍으로 겹치면 밝아지고 엇갈리면 상쇄된다. 이 위상 덧셈이 곧 표면 넓이 위의 적분과 같아진다.

**⚠ 이 식이 채우는 것은 완전한 σ 의 한 단면뿐이다.** RCS 의 일반형은 입사방향과 산란방향의 **네 각도 함수** $\sigma_{\text{RCS}}(\theta_{\text{in}},\phi_{\text{in}};\,\theta_{\text{out}},\phi_{\text{out}})$ 다. 위에서 입사와 산란을 **하나의 $\hat u$** 로 함께 쓴 것은 $\theta_{\text{out}}=\theta_{\text{in}},\ \phi_{\text{out}}=\phi_{\text{in}}$ 인 **후방산란(모노스태틱) 단면 한 장**만 채운다는 뜻이다 — 바이스태틱 일반형(임의 $\hat u_{\text{out}}\neq\hat u_{\text{in}}$)의 경계는 §5·§6 에서 다룬다.

> **직관적으로** — 어두운 방에서 손전등 빔으로 물건을 통째로 덮고(①), 밝게 빛나는 점 하나하나가 내 쪽으로 되쏘는 양을 그 자리에서 계산해 합산한다(②). 그 합이 표적의 밝기다.

그리고 **가림이 공짜로 따라온다.** 광선은 맨 앞에 처음 부딪힌 곳에서 멈추므로, 뒤에 가려진 면은 애초에 합에 끼지 못한다(§5 가 이 이득을 잰다).

**③ 준투명 셸은 통과시켜 안쪽 금속까지 본다.** 드론 후방산란을 실제로 지배하는 건 플라스틱 껍데기가 아니라 그 **안에 든 금속**(배터리 팩·PCB 그라운드플레인)이다. 플라스틱 셸은 반쯤 투명해서(|Γ|≈0.28 → 전력의 ~92% 가 통과) 파가 셸을 지나 내부 금속에 닿고 되돌아 나온다. 그래서 셸에 맞은 광선은 **셸을 지운 장면으로 한 번 더 쏴** 내부 금속을 찾고, 왕복 투과계수 $\tau=1-|\Gamma_{\text{shell}}|^2\;(\approx 0.92)$ 로 세기를 줄여 외부 기여와 **위상 맞춰 합산**한다. 이 단계가 빠지면 광선이 첫 충돌(셸)에서 멈춰 **지배적 산란체인 내부 금속이 통째로 빠지고**(직접 확인: 배터리·PCB 히트 0), 밝기가 약 **1.3 dB 낮게** 나온다 — 재질 모델이 셸 |Γ| 를 0.28 로 낮춘 전제('셸 통과 → 내부 금속 지배') 와 엔진이 어긋나던 자기모순을 이 투과가 없앤다. ⚠ 얇은 셸의 굴절 굴곡·유전체 내부 위상지연은 아직 무시하는 1차 투과 근사다.

핵심은 **새 광선엔진을 만들지 않는다**는 것이다. 광선추적 자체는 **Sionna 의 전파 솔버가 쓰는 Mitsuba 3 / OptiX 엔진을 그대로** GPU 에서 재사용하고(중복계산 없음), 그 위에 선행이 오래 검증해 온 표준 산란적분(자작 PO 표면적분, `src/rcs_sbr.py`)만 얹는다. 스칼라 σ 를 통째로 주입하는 대신 면 조각별 기여를 **복소장 E** 로 더해 **위상을 보존**하므로, 외생 σ 를 넣는 (c) 방식보다 한 단계 앞선다(⚠ 편파는 보존하지 않는다 — E 는 복소 스칼라이고 재질 반사도 스칼라 |Γ| 다). 이 표준 방법이 실제로 맞는 값을 내는지는 §4 에서 답을 아는 물건으로 확인한다.

![.](outputs/renders/anim/paths_build.gif)

<sub>Sionna(Mitsuba) 광선이 반사를 거듭하며 늘어나는 모습 — SBR 은 이 광선으로 표적을 조준한다.</sub>

![송신점→표적→수신점 1회 반사 경로 — 챔버 전경](outputs/renders/rt_10_paths_1bounce_wide.png)

<sub>같은 광선을 챔버 전경에서 본 모습. 송신점(빨강, 왼쪽 벽)에서 나간 전파가 표적에서 꺾여 수신점(초록, 오른쪽 벽)으로 들어오는 **바이스태틱 기하**가 한눈에 보인다. 흡수체 피라미드 벽은 반사를 죽이고, 반사면인 바닥을 스치는 경로가 뒤에서 다룰 **바닥 유령**(report09)의 씨앗이다. SBR 은 정확히 이 표적-조준 광선 다발 위에서 산란적분을 수행한다.</sub>

---
## §4. 검증 — 답을 아는 물건에 대고 재보기

SBR+PO 계산이 옳게 도는지 확인하는 표준 절차는 **답이 이미 알려진 정준 표적(canonical target)에 대고 재보는 것**이다 — Sionna-RT 에 커스텀 산란(UTD) add-on 을 얹는 선행 연구(예: Ziganshin, arXiv:2604.05991)도 구·원통 같은 정준체를 해석해·상용 솔버(FEKO)·실측과 대조해 검증한다. 레이더 교과서는 두 물건의 밝기를 폐형식(닫힌 공식)으로 준다: 정면으로 조사한 **금속 평판**은 σ = 4πA²/λ², **금속구**는 σ = πr².

**검증 조건을 먼저 못박는다.** 이 대조는 **단일 입사방향의 후방산란 한 점**이다 — 구는 az = 0°·el = 0°, 평판은 정면(el = 90°)으로 한 방향만 쏜다(`viz_report2.py` `measure_sbr_validation()`). 방위평균도, 각도응답 곡선의 MAE 도 아니다. 여기서 재는 것은 커널이 **한 방향에서** 옳은 크기를 내느냐이고, 방위 의존성은 아래 드론 수렴표와 report08 이 맡는다.

![sbr validate](outputs/figures/report2_sbr_validate.png)

**@ 3.5 GHz (λ = 8.57 cm) 대조:**

| 표적 | 교과서 값 | SBR 오차 | 우리 설정 λ/16 | report6 재호출 |
|---|---|---|---|---|
| 금속 평판 0.4 × 0.4 m (정면) | +16.42 dBsm | **-0.01 dB** (λ/6) | -0.17 dB | -0.01 dB |
| 금속구 r = 0.5 m | -1.05 dBsm | **+0.39 dB** (λ/10) | -0.58 dB | +0.39 dB |

**이 표를 읽는 법 — 두 줄의 무게가 다르다.**

**① 평판은 '엔진 검증' 이 아니라 '면적 구적(quadrature) 확인' 이다.** 정면입사(el = 90°)에서는 모든 히트의 위상항 $e^{j2k\vec p\cdot\hat u}$ 가 **상수**가 되어 $E = |\Gamma|Nd^2$, $\sigma = (4\pi/\lambda^2)(Nd^2)^2$ 이고 기준 $4\pi A^2/\lambda^2$ 와의 비에서 **λ 가 항등적으로 소거된다.** 즉 이 줄이 재는 것은 '광선격자가 정사각형을 얼마나 잘 타일링하나' 뿐이고, 위상도 파장 의존도 들어가지 않는다. 실제로 오차가 격자밀도에 대해 **비트 단위로 반복**된다 — λ/6·λ/12·λ/30 가 -0.013736 dB 로 동일, λ/8·λ/16 가 -0.169538 dB 로 동일한 정수 정합성 지문이다. 우리 커널 주석도 같은 말을 한다(`rcs_sbr.py`: *평판 정면은 위상항이 상수라 이 진동을 진단 못 한다*). **위상항이 살아 있는 비스듬한 입사로 다시 재는 것은 남은 과제다.**

**② 위상까지 검증하는 줄은 곡면인 구뿐이고, 그 값은 격자밀도에 단조롭지 않다.** λ/10 에서 +0.39 dB 로 잘 맞지만 λ/12 에서는 +1.45 dB, 실제로 드론에 쓰는 λ/16 에서는 -0.58 dB 다. 즉 **'1 dB 이내' 는 특정 격자에서의 값이지 방법의 보증이 아니다** — 실루엣 근처 grazing 광선의 위상 에일리어싱이 남긴 진동이며, 아래 디더 표가 그 정체를 직접 보인다.

**③ report6 원장의 -0.01/+0.39 dB 는 독립 재현이 아니다.** 두 스크립트가 **같은 함수를 같은 인자로** 부른다(`viz_report2.py:588-589` ↔ `viz_verify_sbr.py:107-108`). 결정론적 동일 코드경로라 값이 같은 것은 당연하고, 이것이 보증하는 것은 **다른 실행·다른 파이프라인에서도 같은 값이 나온다는 회귀 재현성**뿐이다. 독립 구현 대조로 읽으면 안 된다.

**단, 매끈한 구는 원래 까다로운 표적이다 — 정직하게 짚는다.** 구는 레이더 쪽으로 **똑바로 되쏘는 지점(정반사점)이 딱 하나뿐**이라, 촘촘한 광선 격자 중 **한 발이 그 점 위에 떨어지느냐**에 따라 값이 흔들립니다. 격자 밀도는 그대로 두고 격자를 살짝 옆으로 옮겨가며 재보면(그림 b):

| 격자 | 정렬만 흔들었을 때 산포 |
|---|---|
| λ/8 | **5.28 dB** |
| λ/12 | **1.37 dB** |
| λ/16 | **1.78 dB** |
| λ/24 | **0.34 dB** |

→ 거친 λ/8 에서는 격자 위치만으로 **5.28 dB** 가 움직이고, 우리 설정 λ/16 에서도 1.78 dB 가 남으며, 촘촘한 λ/24 에서 0.34 dB 로 닫힌다. 여러 오프셋 격자를 평균한 잔차도 같은 방향이다 — λ/8 -1.19 → λ/16 +0.30 → λ/24 +0.02 dB. 즉 **매끈한 곡면의 단일 격자 값은 λ/24 급에서만 안정되고, 우리 격자에서는 정렬 산포 1.78 dB · 디더평균 잔차 +0.30 dB 가 절대레벨에 남아 있다.**

**하지만 드론은 성질이 다릅니다** — 되쏘는 점이 몸통·팔·모터·프로펠러에 잔뜩 흩어져 있어, 방위(각도)를 돌려가며 평균을 내면 **저절로 안정**된다(그림 c):

| 격자 | λ/6 | λ/8 | λ/12 | λ/16 | λ/24 |
|---|---|---|---|---|---|
| LTE 1.8 GHz 방위평균 [dBsm] | -15.45 | -15.91 | -16.73 | -16.86 | -16.86 |
| 5G NR 3.5 GHz 방위평균 [dBsm] | -16.05 | -17.18 | -18.34 | -18.45 | -18.43 |

→ λ/8 부터는 촘촘한 λ/24 값과 **1.25 dB 이내**로 붙는다(가장 거친 λ/6 만 2.38 dB 벗어난다). **드론 측정은 λ/16 로 돌리므로 수렴 구간 안**이다.

정확히 하면 이 수렴은 **촘촘한 방위 샘플(180개)의 평균**에서 성립한다. 같은 드론을 λ/12 와 λ/24 로 각각 다시 계산해 보면(`report_mesh/outputs/mesh_verify.json` I_sbr_subdiv, 방위 24점 · el 15° · 3.5 GHz) **개별 방위각 하나의 값**은 최대 2.00 dB 흔들리고(p95 1.40 · 평균 0.50 dB), 그 방위 24점 **평균**은 0.14 dB 만 움직인다(같은 λ/12→λ/24 를 방위 180점으로 재면 0.09~0.13 dB). 즉 평균이 개별각 진동을 눌러 준다. 그러므로 **개별각 σ 값은 수렴한 숫자가 아니며 인용하지 않는다** — 우리가 밖으로 내보내는 것은 촘촘한 방위평균뿐이다.

덧붙여 삼각형을 4배(28,548→114,192면) 잘게 쪼개도 방위평균 σ 변화는 2.9e-07 dB(개별각 최대 1.3e-05 dB, 같은 파일 subdivision_invariance)다. 다만 이것을 **광선격자 수렴의 증거로 읽어서는 안 된다** — PO 합의 구적점은 삼각형이 아니라 **광선 히트**라서, 면을 중점분할해도 같은 표면·같은 히트가 남는다. 값이 같은 것은 표면이 같으므로 그렇고, 이 검사가 실제로 지키는 것은 **구현이 면 수에 의존하지 않는다**는 것이다(면별 가중이나 면별 지터가 섞여 들어갔다면 깨진다 — `report_mesh/src/verify_mesh_suite.py:376` 이 이 검사를 그 목적으로 적어 둔다). 수치 놉은 삼각형 수가 아니라 위의 광선격자다.

> 정리하면, 정반사점이 하나뿐인 매끈한 구가 이 방법에게 가장 까다로운 표적이고, 되쏘는 점이 많은 드론은 방위평균이 스스로 값을 안정시켜 준다 — 표적의 성질 차이다.

---
## §5. 가림 — SBR 이 순수 PO 대비 실제로 고치는 것

가림을 처리하지 않는 순수 PO 는 표면 조각이 시선 쪽을 향하기만 하면($\hat n\cdot\hat u > 0$) '빛나는 면' 으로 센다 — **앞의 부품에 완전히 가려져 실제로는 레이더에 안 보여도** 그렇다. 반면 SBR 은 광선을 실제로 쏘므로, 뒤에 숨은 면은 광선이 앞면에서 멈춰 애초에 닿지 못한다. 이 가림 처리는 상용 EM 솔버의 SBR 과 같은 성질이다.

![occlusion](outputs/figures/report2_occlusion.png)

DJI Mavic 4 Pro 를 한 방위(방위각 30°, 올려본각 15°)에서 들여다보면:

| | 조각 수 | 투영 넓이 |
|---|---|---|
| 순수 PO 가 '빛난다' 고 센 면 | **19,990** | 975 cm² |
| └ 그중 **실제로는 뒤에 가려진** 면 | **7,695 (38%)** | — |
| 광선이 **실제로 맞은** 면 (SBR) | 12,295 | **492 cm²** |

→ 순수 PO 는 실제로 보이는 것보다 **약 2.0 배 넓은 면적**을 밝기에 넣고 있었다. 가려진 면을 빼면 밝기가 그만큼 내려간다:

**DJI Mavic 4 Pro, 72 방위 평균, 올려본각 15°, 3.5 GHz:**

| 방식 | 방위평균 밝기 | 차이 |
|---|---|---|
| 순수 PO (가림 없음) | -15.21 dBsm | — |
| **SBR (가림 처리)** | **-19.56 dBsm** | **-4.35 dB** ← 가림 |
| SBR + 오목부 다중반사 | -19.44 dBsm | +0.11 dB ← 다중반사 |

![po vs sbr](outputs/figures/report2_po_vs_sbr.png)

흥미롭게도, PO 계열에 흔히 지적되는 '오목한 곳의 다중반사를 놓친다' 는 약점은 여기서 **+0.11 dB** 짜리 사소한 항이었다. 실제로 밝기를 바꾼 것은 **가림(4.35 dB)** 이었다 — 한 자릿수 차이다.

**단, 가림 대가의 절대 dB 는 샘플링에 따라 갈린다 — 그래서 방향과 규모만 주장한다.** 같은 기체·같은 올려본각을 조건만 바꿔(방위 36점, SBR 광선격자 λ/12) 다시 재면 -15.89 → -18.60 dBsm 으로 가림이 -2.71 dB 다(`report6_sbr.json compare`). 위 표(72점, λ/16)의 -4.35 dB 와 1.64 dB 차이가 나며, 이는 §4 가 보인 광선격자·방위샘플 민감도와 같은 급이다. 우리가 주장하는 것은 **가림이 σ 를 내린다는 방향과 수 dB 급 규모**이지 특정 dB 값이 아니다.

> 이것이 SBR 이 순수 PO 대비 실제로 고치는 지점이다. 광선을 쏘아 **레이더에 실제로 보이는 면만** 밝기에 넣으니, 뒤에 숨은 면까지 세는 순수 PO 보다 값이 낮아진다.

**⚠ 스코프 — 여기서 다루는 가림도, 위의 σ 도 전부 '모노스태틱' 이다.** 이 절의 가시성 판정은 송·수신이 같은 방향($\hat u_s = \hat u_i$)이라는 전제 위에 서 있다. 우리 프로젝트는 패시브 **바이스태틱**이므로 이 전제는 자동으로 이어지지 않는다 — 조명원 쪽에서 보이는 면과 수신기 쪽에서 보이는 면이 갈리기 때문이다. 바이스태틱 일반형(`rcs_sbr.py` `rcs_sbr_multistatic()`)은 구현되어 있고 그 독스트링이 적용범위를 명시한다: 전방산란(β→180°)에서는 조명 게이트와 수신 게이트가 상호배타가 되어 **σ ≡ 0** 이 되고(그림자 복사 = Babinet 전방로브를 lit-PO 가 못 낸다), 비볼록 표적의 깊은 널에서는 **상반성 σ(û_i,û_s) = σ(û_s,û_i) 가 깨진다**. 따라서 이 리포트의 σ 는 **모노스태틱 등가값**으로 읽어야 하며, 바이스태틱 결론(report12)으로 그대로 상속되지 않는다.

---
## §6. 선행이 우리 방법을 비판한다 — 회절 항과 이산화 기준

§5 가 스스로 밝힌 한계(전방산란 σ≡0 · 상반성 부분성립 · 모노스태틱 등가)는 우연이 아니라 **SBR+PO 라는 방법 계열이 원래 갖는 경계**다. 같은 'Sionna-RT 확장' 계열에서 우리와 가장 가까운 선행인 Ziganshin(arXiv:2604.05991)은 바로 이 경계를 겨냥해 **SBR+PO 를 자기 방법(Sionna-RT + UTD 회절)의 대척점으로 명시적으로 비판**한다. 원문 그대로:

> *"This SBR+PO approach ... is limited to the illuminated region and is not suitable to predict the scattered field in the shadow region of the obstacle. Furthermore, the need to cascade PO after RT negates the computational advantages of RT."* — Ziganshin §I (arXiv:2604.05991)

이 비판은 **절반은 정확하다.** §5 가 이미 자발적으로 공개했듯, lit-PO 는 **조명된 영역만** 적분하므로 그림자 영역의 회절장(전방로브 = Babinet)을 못 내고 전방산란에서 σ≡0 이 된다. 우리에게 빠져 있던 것은 **물리 인식이 아니라 이 선행을 인용해 그 경계를 명명하는 프레이밍**이었다 — 그 물리 자체는 `rcs_sbr.py` `rcs_sbr_multistatic()` 독스트링과 §5 가 이미 적고 있었다. (덧붙여 Ziganshin 은 자기 future work 로 *"time-varying substructures (micro-Doppler)"* 와 *"drones"* 를 지목하므로, 두 방법의 적용 영역이 겹치는 것도 사실이다.)

### 전계 항목 원장 — 무엇을 채우고 무엇을 비우나

Ziganshin 은 표적 산란장을 **네 항의 합**으로 분해한다. 우리 SBR+PO 가 어느 항을 채우는지 그 원장에 나란히 놓으면 우리 방법의 경계가 한눈에 보인다:

$$E_{\text{total}} = E_{\text{direct}} + \sum E_{\text{specular}} + \sum E_{\text{edge}} + \sum E_{\text{vertex}}$$

| 항 | 물리 | 우리 SBR+PO | Ziganshin RT+UTD |
|---|---|---|---|
| $E_{\text{direct}}$ | 표적을 조명하는 입사장 / 직접경로 | ✅ (Sionna 광선 = `h_direct`) | ✅ |
| $\sum E_{\text{specular}}$ | 표면 정반사·다중반사 (PO 표면적분) | ✅ **채운다** (§3·§5) | ✅ |
| $\sum E_{\text{edge}}$ | 모서리 회절 (UTD wedge) | ❌ **비운다** | ✅ |
| $\sum E_{\text{vertex}}$ | 정점(꼭짓점) 회절 | ❌ **비운다** | ✅ |

즉 우리는 **정반사 항(specular)** 을 위상까지 맞춰 채우고 **회절 두 항(edge · vertex)은 비운다.** §5 가 자발적으로 공개한 '그림자 영역·깊은 널 부정확' 은 정확히 이 두 빈 항의 결과다.

### 우리 반론 — few-λ 유전체 드론에서는 UTD 유효조건이 성립하지 않는다

그러나 **그들의 처방(UTD 로 회절 항을 채우기)이 우리 표적에는 유효하지 않다.** Ziganshin 의 UTD 는 **facet 이 파장보다 충분히 커야**(유효 하한 E > 1.5λ) 성립하는 고주파 근사이고, 그들의 검증 대상은 **PEC 차량 · 구 · 원기둥**(2–10 GHz, 차량 facet 의 E²/(Rλ) ≈ 0.4–0.6)이다. 우리 표적은 **부위별 유전체 소형 드론**으로, 기체 대각이 28~104 cm 이고 3.5 GHz 에서 UTD 유효 하한은 1.5λ ≈ 13 cm 다. 프로펠러 시위 · 암 폭 · 모터 · 랜딩기어 같은 **개별 산란 특징의 대부분이 이 1.5λ 하한보다 작아**(공진영역), 회절 두 항을 UTD 로 채우는 것도 우리 영역에서는 원리적으로 정당화되지 않는다. 소형 유전체 드론의 few-λ 회절은 UTD 가 아니라 full-wave(MoM/FEKO)나 측정으로 앵커링해야 하며(report08), 그것이 이 프로젝트의 열린 과제다. (⚠ 이는 우리 lit-PO 가 회절을 낸다는 뜻이 아니다 — 우리도 그 두 항은 비어 있고, 다만 그들의 대안조차 이 영역에선 유효하지 않다는 것이다.)

### 이산화 품질 — E²/(Rλ)

Ziganshin 은 곡면을 facet 으로 쪼갤 때의 품질을 무차원 지표 **E²/(Rλ)** (E = facet 크기, R = 국소 곡률반경, λ = 파장)로 재고, 후방산란 정확도가 이 값이 작을수록 좋아지되 **≈0.5 부근에서 수렴(개선이 미미해지는 무릎)** 한다고 보고한다(자기 차량 facet ≈ 0.4–0.6). 원문은 최적 구간이 응용에 따라 달라 경험적으로 정해야 한다고 밝히므로 ≈0.5 는 보편 임계값이 아니라 그들의 수렴 무릎이다. 우리 PO 는 삼각형이 아니라 **광선 히트**를 구적점으로 쓰므로(§4 세분 불변성이 이를 직접 보인다), 실질 이산화 눈금은 삼각형 크기가 아니라 **광선격자 간격**이다 — 3.5 GHz 에서 λ/16 ≈ 5.4 mm(수렴 연구는 λ/12 ≈ 7.14 mm → λ/24). 이 몇 mm 격자가 드론 facet 크기와 **같은 자릿수**이므로 **수치 병목은 삼각형 수가 아니라 광선격자 쪽**이다(§4 가 삼각형을 4배 세분해도 σ 가 사실상 불변임을 이미 보였다 — 놉은 격자다).

⚠ **우리 5기종 메쉬의 E²/(Rλ) 값 자체는 아직 산출하지 않았다** — facet 별 모서리 길이 대 국소 곡률반경을 재는 이 지표는 현재 원장(`mesh_verify.json`)에 없고 **다음 단계에서 계산**한다. 지금 확정적으로 말할 수 있는 것은 **UTD 유효 하한 E > 1.5λ 가 우리 few-λ 특징에서 성립하지 않는다**는 정성적 사실까지다.

In [ ]:
# §4·§5 재현 — SBR 커널 검증(교과서 값) + 가림 대조
import rcs_sbr
rcs_sbr.validate(3.5e9)        # 금속구 πr² / 평판 4πA²/λ² 대조 + 격자 수렴
rcs_sbr.compare_with_po()      # 순수 PO vs SBR (가림의 대가)

In [ ]:
# SBR 커널을 한 방향에서 직접 호출 — 광선 격자를 쏘고(가림 처리), 위상 맞춰 더해 밝기를 낸다.
import numpy as np
from rcs_po import drone_rcs_pattern_bw, dbsm    # 기본 엔진이 'sbr' 이다

az = np.arange(0, 361, 2.0)
sig, n_rays = drone_rcs_pattern_bw('mavic4pro', 3.5e9, 100e6, az, el_deg=15.0, n_f=5)
print(f'방위당 광선 {n_rays:,}발  (격자 λ/16)')
print(f'방위평균 밝기 {dbsm(np.mean(sig)):+.2f} dBsm')
# ↑ 드론별 밝기 '결과'의 해석은 다음 리포트(report08) 소관이다.

---
## 정리

1. **밝기(RCS)는 탐지에 필수**인데, Sionna RT `PathSolver` 는 산란적분(∫) 단계가 없어 σ 를 주지 못한다(§1, 증거는 report06). 이 공백을 ISAC 문헌은 세 갈래로 메우고(§2), 우리는 **(d) 자작 SBR+PO** 를 택했다.
2. **방법:** 표적을 정면 조준해 촘촘한 평행 광선으로 덮고, 맞은 점마다 레이더로 되쏘는 양을 위상까지 맞춰 더한다(§3). 광선추적은 **Sionna 가 쓰는 Mitsuba 엔진을 그대로 재사용**하고 그 위에 자작 PO 표면적분만 얹는다 — GPU BVH SBR+PO(arXiv:2604.09243)와 같은 계열이다(적용범위 차이는 report06 §4).
3. **옳은가?** 단일 입사방향에서 금속 평판(-0.01 dB @λ/6)·금속구(+0.39 dB @λ/10)이 교과서 값과 1 dB 이내였다(§4). 단 **평판 정면은 위상항이 상수라 면적 구적 확인**이고, 위상까지 거는 구는 격자밀도에 단조롭지 않다(λ/12 에서 +1.45 dB, 우리 λ/16 에서 -0.58 dB). report6 원장의 같은 값은 **같은 함수·같은 인자 재호출**이라 회귀 확인이다. 드론 쪽은 방위평균이 저절로 안정돼 λ/16 가 수렴 구간 안이다.
4. **무엇을 고치나?** 순수 PO 는 뒤에 숨어 안 보이는 면(이 방위에서 38%)까지 세어 밝기를 부풀렸다. 가려진 면을 빼자 밝기가 **-15.21 → -19.56 dBsm** 으로 내려갔다 — 다중반사가 아니라 **가림이 진짜 이득**이었다(§5).

**이 리포트가 보장하지 않는 것:** **절대 RCS 값**(정준 표적 해석해로만 검증·드론 실측 앵커 없음 → report08 — 방법의 정합성과 가림의 방향·규모만 주장), **매끈한 구의 단일 격자 값**·**다중반사의 정확한 dB**(인용 금지), **위상·파장에 대한 평판 검증**(정면입사라 λ 가 소거된다 — 비스듬한 입사 재측정은 남은 과제), **바이스태틱 σ**(여기 값은 모노스태틱 등가이고 전방산란·상반성 한계는 §5 에 명시), **모서리·정점 회절**(E_edge·E_vertex 두 항을 비운다 — §6; 선행의 UTD 처방도 유효 하한 E>1.5λ 가 few-λ 드론에서 불성립), **유전체 셸의 2차 효과**(셸 투과는 넣었지만 굴절 굴곡·내부 위상지연을 무시하는 1차 근사다).

> **다음 리포트**: [report08](report08.ipynb) — 이 SBR 로 잰 **실제 드론 5종의 밝기(RCS) 결과**. 절대값은 실측 문헌 RCS 로 앵커링한다.